In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense,
                                     Input, RepeatVector, TimeDistributed)
from tensorflow.keras.utils import to_categorical

# ════════════════════════════════════════════
#  TASK 1 — LSTM Text Classification (IMDB)
# ════════════════════════════════════════════

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
x_train = pad_sequences(x_train, maxlen=200)
x_test  = pad_sequences(x_test,  maxlen=200)

clf = Sequential([
    Embedding(10000, 64, input_length=200),
    LSTM(64),
    Dense(1, activation='sigmoid')
])
clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
clf.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.2, verbose=1)

loss, acc = clf.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc*100:.2f}%")

# ════════════════════════════════════════════
#  TASK 2 — Seq2Seq (Number → Words)
#  e.g.  "123" → "one two three"
# ════════════════════════════════════════════

WORDS = ['zero','one','two','three','four','five','six','seven','eight','nine']

def generate(n=20000):
    X, Y = [], []
    for _ in range(n):
        nums = [np.random.randint(0, 10) for _ in range(3)]
        X.append(nums)
        Y.append([w[0] for w in [WORDS[d] for d in nums]])  # first char of each word
    return np.array(X), Y

X_data, Y_raw = generate()

# Encode targets as character sequences
all_chars = sorted(set(c for seq in Y_raw for c in seq)) + ['<PAD>']
c2i = {c: i for i, c in enumerate(all_chars)}
VOCAB = len(c2i)
MAX_OUT = max(len(s) for s in Y_raw)

Y_data = np.array([[c2i[c] for c in s] + [c2i['<PAD>']]*(MAX_OUT-len(s))
                   for s in Y_raw])
Y_hot  = to_categorical(Y_data, num_classes=VOCAB)

# Encoder
enc_in  = Input(shape=(3,))
enc_emb = Embedding(10, 32)(enc_in)
enc_out = LSTM(64)(enc_emb)

# Decoder (simple: repeat encoder state → decode)
dec     = RepeatVector(MAX_OUT)(enc_out)
dec     = LSTM(64, return_sequences=True)(dec)
dec_out = TimeDistributed(Dense(VOCAB, activation='softmax'))(dec)

seq2seq = Model(enc_in, dec_out)
seq2seq.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
seq2seq.fit(X_data, Y_hot, epochs=20, batch_size=256, validation_split=0.1, verbose=1)

# Test
def predict(nums):
    x   = np.array([nums])
    out = seq2seq.predict(x, verbose=0)[0]
    return ''.join(all_chars[np.argmax(t)] for t in out).replace('<PAD>', '')

print("\nSeq2Seq Predictions:")
for test in [[1,2,3], [4,0,9], [7,5,2]]:
    print(f"  {test} → '{predict(test)}'  (expected: '{' '.join(WORDS[d][0] for d in test)}')")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 53s 326ms/step - accuracy: 0.7786 - loss: 0.4471 - val_accuracy: 0.8516 - val_loss: 0.3491
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 322ms/step - accuracy: 0.8996 - loss: 0.2609 - val_accuracy: 0.8684 - val_loss: 0.3117
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 50s 321ms/step - accuracy: 0.9301 - loss: 0.1885 - val_accuracy: 0.8694 - val_loss: 0.3205
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 50s 320ms/step - accuracy: 0.9515 - loss: 0.1370 - val_accuracy: 0.8604 - val_loss: 0.3328
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 49s 312ms/step - accuracy: 0.9649 - loss: 0.1058 - val_accuracy: 0.8656 - val_loss: 0.3797

Test Accuracy: 85.79%
Epoch 1/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.3692 - loss: 1.7387 - val_accuracy: 0.4723 - val_loss: 1.2920
Epoch 2/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5379 - loss: 1.0170 - val_accuracy: 0.6495 - val_loss: 0.7803
Epoch 3/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7806 - l